# Airbnb Geospatial Analysis — Data Cleaning

## Objective
This notebook loads and cleans the Airbnb Albany dataset for exploratory analysis and geospatial hotspot detection.

## Tasks
- Load datasets
- Inspect schema and missing values
- Clean pricing columns
- Convert dates
- Handle duplicates and invalid coordinates
- Save processed datasets

In [71]:
import pandas as pd
import numpy as np
import ast

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [72]:
listings = pd.read_csv("../data/raw/listings.csv.gz")
calendar = pd.read_csv("../data/raw/calendar.csv.gz")
reviews = pd.read_csv("../data/raw/reviews.csv.gz")

In [73]:
print("Listings Shape:", listings.shape)
print("Calendar Shape:", calendar.shape)
print("Reviews Shape:", reviews.shape)

Listings Shape: (453, 85)
Calendar Shape: (165345, 7)
Reviews Shape: (27892, 6)


In [74]:
listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 453 entries, 0 to 452
Data columns (total 85 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            453 non-null    int64  
 1   listing_url                                   453 non-null    str    
 2   scrape_id                                     453 non-null    int64  
 3   last_scraped                                  453 non-null    str    
 4   source                                        453 non-null    str    
 5   name                                          453 non-null    str    
 6   description                                   442 non-null    str    
 7   neighborhood_overview                         0 non-null      float64
 8   picture_url                                   453 non-null    str    
 9   host_id                                       453 non-null    int64  
 10  h

In [75]:
calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 165345 entries, 0 to 165344
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   listing_id      165345 non-null  int64  
 1   date            165345 non-null  str    
 2   available       165345 non-null  str    
 3   price           0 non-null       float64
 4   adjusted_price  0 non-null       float64
 5   minimum_nights  165345 non-null  int64  
 6   maximum_nights  165345 non-null  int64  
dtypes: float64(2), int64(3), str(2)
memory usage: 8.8 MB


In [76]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 27892 entries, 0 to 27891
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   listing_id     27892 non-null  int64
 1   id             27892 non-null  int64
 2   date           27892 non-null  str  
 3   reviewer_id    27892 non-null  int64
 4   reviewer_name  27892 non-null  str  
 5   comments       27884 non-null  str  
dtypes: int64(3), str(3)
memory usage: 1.3 MB


In [77]:
missing_values = listings.isnull().sum().sort_values(ascending=False)

missing_values[missing_values > 0]

neighborhood_overview           453
host_since                      453
host_acceptance_rate            453
host_response_rate              453
host_response_time              453
host_total_listings_count       453
host_verifications              453
license                         453
instant_bookable                453
calendar_updated                453
estimated_revenue_l365d         453
price                           453
neighbourhood                   453
host_thumbnail_url              453
host_neighbourhood              453
neighbourhood_group_cleansed    453
host_about                      199
host_location                   120
review_scores_value              58
reviews_per_month                58
review_scores_cleanliness        58
review_scores_location           58
review_scores_checkin            58
first_review                     58
last_review                      58
review_scores_rating             58
review_scores_communication      58
review_scores_accuracy      

In [78]:
for i in [listings,calendar,reviews]:
    
    missing_percentage = (
        i.isnull().mean() * 100
    ).sort_values(ascending=False)

    print(missing_percentage[missing_percentage > 0])

neighborhood_overview           100.000000
host_since                      100.000000
host_acceptance_rate            100.000000
host_response_rate              100.000000
host_response_time              100.000000
host_total_listings_count       100.000000
host_verifications              100.000000
license                         100.000000
instant_bookable                100.000000
calendar_updated                100.000000
estimated_revenue_l365d         100.000000
price                           100.000000
neighbourhood                   100.000000
host_thumbnail_url              100.000000
host_neighbourhood              100.000000
neighbourhood_group_cleansed    100.000000
host_about                       43.929360
host_location                    26.490066
review_scores_value              12.803532
reviews_per_month                12.803532
review_scores_cleanliness        12.803532
review_scores_location           12.803532
review_scores_checkin            12.803532
first_revie

In [79]:
columns_to_drop = [
    'neighborhood_overview',
    'host_since',
    'host_acceptance_rate',
    'host_response_rate',
    'host_response_time',
    'host_total_listings_count',
    'host_verifications',
    'license',
    'instant_bookable',
    'calendar_updated',
    'estimated_revenue_l365d',
    'price',
    'neighbourhood',
    'host_thumbnail_url',
    'host_neighbourhood',
    'neighbourhood_group_cleansed'
]

listings = listings.drop(columns=columns_to_drop)

In [80]:
important_columns = [
    'id',
    'host_id',
    'host_name',
    'host_location',
    'host_is_superhost',

    'neighbourhood_cleansed',

    'latitude',
    'longitude',

    'property_type',
    'room_type',

    'accommodates',
    'bathrooms',
    'bathrooms_text',
    'bedrooms',
    'beds',

    'amenities',

    'minimum_nights',
    'maximum_nights',

    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',

    'number_of_reviews',
    'number_of_reviews_ltm',
    'number_of_reviews_l30d',

    'estimated_occupancy_l365d',

    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',

    'reviews_per_month',

    'first_review',
    'last_review',

    'calculated_host_listings_count',

    'host_has_profile_pic',
    'host_identity_verified',
    'has_availability'
]

listings = listings[important_columns]

In [81]:
listings = listings.rename(columns={
    'id': 'listing_id',
    'neighbourhood_cleansed': 'neighborhood'
})

In [82]:
date_columns = [
    'first_review',
    'last_review'
]

for col in date_columns:
    listings[col] = pd.to_datetime(
        listings[col],
        format='%Y-%m-%d',
        errors='coerce'
    )


In [83]:
bool_columns = [
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    'has_availability'
]

for col in bool_columns:
    listings[col] = listings[col].map({
        't': True,
        'f': False
    })

In [84]:
listings['occupancy_rate'] = (
    listings['estimated_occupancy_l365d'] / 365
)

In [85]:
listings['amenities_count'] = listings['amenities'].apply(
    lambda x: len(ast.literal_eval(x))
    if pd.notnull(x) else 0
)

In [86]:
listings = listings[
    listings['latitude'].between(-90, 90) &
    listings['longitude'].between(-180, 180)
]

In [87]:
calendar = calendar.drop(columns=[
    'price',
    'adjusted_price'
])

In [88]:
calendar['date'] = pd.to_datetime(
    calendar['date'],
    format='%Y-%m-%d',
    errors='coerce'
)

In [89]:
calendar['available'] = calendar['available'].map({
    't': True,
    'f': False
})

In [90]:
availability_summary = (
    calendar
    .groupby('listing_id')['available']
    .mean()
    .reset_index()
)

availability_summary = availability_summary.rename(columns={
    'available': 'availability_ratio'
})

In [91]:
listings = listings.merge(
    availability_summary,
    on='listing_id',
    how='left'
)

In [92]:
listings['estimated_occupancy_from_calendar'] = (
    1 - listings['availability_ratio']
)

In [93]:
reviews = reviews.dropna(subset=['comments'])

In [94]:
reviews['date'] = pd.to_datetime(
    reviews['date'],
    format='%Y-%m-%d',
    errors='coerce'
)

In [95]:
reviews['comments'] = (
    reviews['comments']
    .str.lower()
    .str.strip()
)

In [96]:
neighborhood_summary = (
    listings
    .groupby('neighborhood')
    .agg({
        'listing_id': 'count',
        'occupancy_rate': 'mean',
        'estimated_occupancy_from_calendar': 'mean',
        'number_of_reviews': 'mean',
        'reviews_per_month': 'mean',
        'review_scores_rating': 'mean',
        'calculated_host_listings_count': 'mean'
    })
    .rename(columns={
        'listing_id': 'listing_count'
    })
    .reset_index()
)

In [97]:
neighborhood_summary.head()

,neighborhood,listing_count,occupancy_rate,estimated_occupancy_from_calendar,number_of_reviews,reviews_per_month,review_scores_rating,calculated_host_listings_count
0,EIGHTH WARD,12,0.229909,0.353425,73.750000,3.257143,4.942857,2.000000
1,ELEVENTH WARD,30,0.152694,0.100639,30.966667,1.648000,4.450800,19.966667
2,FIFTEENTH WARD,15,0.330594,0.134795,81.733333,1.870000,4.911333,4.066667
3,FIFTH WARD,19,0.084787,0.138717,16.263158,0.933125,4.368125,18.000000
4,FIRST WARD,9,0.234703,0.358295,44.111111,1.506250,4.692500,2.555556


In [98]:
listings.to_csv(
    "../data/processed/listings_clean.csv",
    index=False
)

In [99]:
calendar.to_csv(
    "../data/processed/calendar_clean.csv",
    index=False
)

In [100]:
reviews.to_csv(
    "../data/processed/reviews_clean.csv",
    index=False
)

In [101]:
neighborhood_summary.to_csv(
    "../data/processed/neighborhood_summary.csv",
    index=False
)

In [102]:
listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 453 entries, 0 to 452
Data columns (total 44 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   listing_id                         453 non-null    int64         
 1   host_id                            453 non-null    int64         
 2   host_name                          453 non-null    str           
 3   host_location                      333 non-null    str           
 4   host_is_superhost                  453 non-null    bool          
 5   neighborhood                       453 non-null    str           
 6   latitude                           453 non-null    float64       
 7   longitude                          453 non-null    float64       
 8   property_type                      453 non-null    str           
 9   room_type                          453 non-null    str           
 10  accommodates                       453 non-null  

In [103]:
listings.describe()

,listing_id,host_id,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,estimated_occupancy_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,reviews_per_month,first_review,last_review,calculated_host_listings_count,occupancy_rate,amenities_count,availability_ratio,estimated_occupancy_from_calendar
count,4.530000e+02,4.530000e+02,453.000000,453.000000,453.000000,437.000000,450.000000,437.00000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,453.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395.000000,395,395,453.000000,453.000000,453.000000,453.000000,453.000000
mean,8.981917e+17,2.505305e+08,42.659237,-73.776699,3.439294,1.231121,1.597778,1.82151,5.825607,631.823400,18.150110,40.211921,63.326711,258.172185,61.571744,13.865342,0.704194,79.620309,4.733291,4.771139,4.749671,4.829342,4.838759,4.633924,4.692835,1.840684,2023-04-02 03:31:26.582278,2025-10-10 12:56:30.379746,7.662252,0.218138,37.379691,0.707321,0.292679
min,2.992450e+06,6.490680e+05,42.630660,-73.876490,1.000000,0.000000,0.000000,0.00000,1.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.500000,2.330000,2.330000,2.500000,1.000000,2.000000,2.500000,0.030000,2014-07-01 00:00:00,2022-08-17 00:00:00,1.000000,0.000000,6.000000,0.000000,0.000000
25%,5.932721e+17,4.762598e+07,42.652620,-73.787714,2.000000,1.000000,1.000000,1.00000,1.000000,365.000000,11.000000,26.000000,46.000000,175.000000,3.000000,1.000000,0.000000,6.000000,4.670000,4.750000,4.685000,4.810000,4.840000,4.500000,4.630000,0.425000,2022-01-05 12:00:00,2025-09-29 12:00:00,1.000000,0.016438,25.000000,0.479452,0.021918
50%,1.020519e+18,2.329679e+08,42.657620,-73.774560,2.000000,1.000000,1.000000,1.00000,1.000000,365.000000,22.000000,48.000000,76.000000,307.000000,18.000000,6.000000,0.000000,54.000000,4.850000,4.900000,4.870000,4.940000,4.950000,4.790000,4.810000,1.230000,2023-11-03 00:00:00,2025-12-21 00:00:00,5.000000,0.147945,39.000000,0.841096,0.158904
75%,1.383911e+18,4.330961e+08,42.666120,-73.763820,4.000000,1.000000,2.000000,2.00000,3.000000,1125.000000,27.000000,57.000000,87.000000,357.000000,68.000000,18.000000,1.000000,126.000000,4.970000,4.980000,4.980000,5.000000,5.000000,4.930000,4.930000,2.575000,2025-05-01 12:00:00,2026-01-28 00:00:00,11.000000,0.345205,47.000000,0.978082,0.520548
max,1.611096e+18,7.395856e+08,42.714900,-73.738250,16.000000,7.000000,9.000000,10.00000,180.000000,1125.000000,30.000000,60.000000,90.000000,365.000000,972.000000,127.000000,9.000000,255.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,10.440000,2026-01-31 00:00:00,2026-02-14 00:00:00,30.000000,0.698630,84.000000,1.000000,1.000000
std,5.701275e+17,2.013526e+08,0.010570,0.018555,2.436622,0.639231,1.171485,1.32381,14.542524,428.346775,10.458704,20.146678,28.941031,115.813576,114.667091,19.267722,1.363528,84.195552,0.382619,0.378788,0.368865,0.329483,0.364384,0.451638,0.403480,1.888654,NaN,NaN,7.994650,0.230673,15.239583,0.317297,0.317297
